<a href="https://colab.research.google.com/github/Ansh1657/MRI-Deepfake-Detection-System/blob/main/notebooks/01_skull_stripping_rough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports & Setup

In [ ]:
"""
Skull Stripping R&D Notebook (Exploratory)
Uses Gentle Erosion + Component Selection to isolate brain tissue.
Note: This was tested but ultimately bypassed in the final Gradio pipeline
to preserve border-edge GAN artifacts.
"""

import os
import numpy as np
from pathlib import Path
from PIL import Image
import cv2
from scipy import ndimage
import matplotlib.pyplot as plt

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

Core OpenCV Logic

In [ ]:
def get_brain_mask(img_gray: np.ndarray) -> np.ndarray:
    """Generate a binary brain mask from a 2D grayscale MRI slice."""
    # 1. Normalize & Blur
    img_norm = cv2.normalize(img_gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_norm, (5, 5), 0)

    # 2. Otsu threshold
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Gentle Erosion (Break the skull ring)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    eroded = cv2.erode(binary, kernel, iterations=2)

    # 4. Keep largest connected component (The Brain)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(eroded, connectivity=8)
    if num_labels <= 1:
        return np.ones_like(img_gray, dtype=np.uint8) * 255 # Fallback

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    brain_seed = (labels == largest_label).astype(np.uint8) * 255

    # 5. Dilation (Restore original brain size)
    brain_mask = cv2.dilate(brain_seed, kernel, iterations=2)

    # 6. Fill internal holes
    brain_mask = ndimage.binary_fill_holes(brain_mask > 0).astype(np.uint8) * 255

    # 7. Final edge smoothing
    brain_mask = cv2.GaussianBlur(brain_mask, (5, 5), 0)
    _, brain_mask = cv2.threshold(brain_mask, 127, 255, cv2.THRESH_BINARY)

    return brain_mask

def apply_mask(img_gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Apply binary mask to image. Pixels outside mask → 0 (black)."""
    result = img_gray.copy()
    result[mask == 0] = 0
    return result

def skull_strip_image(img_path: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Full pipeline for a single image."""
    img = Image.open(img_path).convert("L")  # Force grayscale
    original = np.array(img, dtype=np.uint8)

    mask = get_brain_mask(original)
    stripped = apply_mask(original, mask)

    return original, mask, stripped

Diagnostics & Visualization

In [ ]:
def diagnose_mask(mask: np.ndarray, img_name: str = "") -> dict:
    total = mask.size
    brain_pixels = np.sum(mask > 0)
    coverage = 100.0 * brain_pixels / total

    result = {
        "name": img_name,
        "coverage_pct": round(coverage, 1),
        "is_empty": coverage < 10,
        "is_full": coverage > 90,
        "status": "OK"
    }

    if result["is_empty"]: result["status"] = "WARN: mask too small"
    elif result["is_full"]: result["status"] = "WARN: mask too large"
    return result

def preview_single(img_path: str):
    """Show original vs mask vs stripped for one image."""
    original, mask, stripped = skull_strip_image(img_path)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ["Original", "Brain mask", "Skull stripped"]
    imgs   = [original, mask, stripped]

    for ax, title, img in zip(axes, titles, imgs):
        ax.imshow(img, cmap="gray")
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle(Path(img_path).name, fontsize=10)
    plt.tight_layout()
    plt.show()

Experiment Execution

In [ ]:
# Select a sample image from the raw dataset to test the OpenCV pipeline
sample_image_path = "../data/raw_mri/real_bright/sample_001.jpg"

if os.path.exists(sample_image_path):
    print("Running Skull Stripping R&D test...")
    preview_single(sample_image_path)

    # Run diagnostics
    _, mask, _ = skull_strip_image(sample_image_path)
    diag = diagnose_mask(mask, "sample_001.jpg")
    print(f"Coverage: {diag['coverage_pct']}% | Status: {diag['status']}")
else:
    print(f"File not found: {sample_image_path}")
    print("Note: Raw IP datasets are withheld from this public repository.")